# Notebook 02 — Scaled Dot-Product Attention

## Learning Objectives
- Understand *why* we scale by 1/√d_k: prevent softmax saturation.
- Use the `ScaledDotProductAttention` module from `src.models.attention`.
- Observe the effect of scaling on attention weight distributions.
- Apply a **causal mask** to prevent attending to future tokens.
- Print and verify tensor shapes at every step.

## Big Picture
Scaled dot-product attention is the atomic operation inside every Transformer.
All the complexity of multi-head attention, encoder blocks, and decoders
ultimately reduces to this one function:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

## 1. Imports and Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

from src.utils.seed import seed_everything
from src.utils.device import get_best_device
from src.utils.cleanup import prepare_run_dir
from src.utils.reporting import save_table_csv
from src.utils.paths import RUNS_ATTENTION_DIR
from src.models.attention import ScaledDotProductAttention

seed_everything(42)
device = get_best_device()
prepare_run_dir(RUNS_ATTENTION_DIR, clean=False)
print(f"Device: {device}")
print("All imports successful.")

## 2. Dataset: Synthetic Q, K, V Tensors

We create small random Q, K, V tensors with controlled shapes to study the attention formula.
In a real Transformer, these are produced by learned linear projections — but the math is identical.

In [ ]:
# Hyperparameters
BATCH_SIZE = 2
NUM_HEADS  = 1   # single head for clear illustration
SEQ_LEN    = 5
D_K        = 8   # key/query dimension
D_V        = 8   # value dimension

torch.manual_seed(42)

# Q, K, V tensors — shape [batch, heads, seq_len, d_k]
Q = torch.randn(BATCH_SIZE, NUM_HEADS, SEQ_LEN, D_K).to(device)
K = torch.randn(BATCH_SIZE, NUM_HEADS, SEQ_LEN, D_K).to(device)
V = torch.randn(BATCH_SIZE, NUM_HEADS, SEQ_LEN, D_V).to(device)

print(f"Q shape: {Q.shape}  (batch, heads, seq_len, d_k)")
print(f"K shape: {K.shape}  (batch, heads, seq_len, d_k)")
print(f"V shape: {V.shape}  (batch, heads, seq_len, d_v)")

## 3. Architecture (Text Diagram)

```
  Q [batch, heads, seq_q, d_k]
  K [batch, heads, seq_k, d_k]
  V [batch, heads, seq_k, d_v]
           │
           ▼
  scores = Q @ K^T            → [batch, heads, seq_q, seq_k]
           │
           ▼
  scaled_scores = scores / √d_k
           │
           ▼ (optional causal mask)
  masked_scores = fill -∞ where mask=True
           │
           ▼
  weights = softmax(scaled_scores)  → [batch, heads, seq_q, seq_k]
           │
           ▼
  output = weights @ V        → [batch, heads, seq_q, d_v]
```

## 4. Theory: Why Scale by √d_k?

Suppose Q and K have entries sampled from N(0, 1). Then Q·K has variance d_k.
As d_k grows, the dot products get larger in magnitude, which pushes softmax
into its flat regions (near 0 or 1), causing very small gradients.

**Fix**: divide by √d_k → variance becomes 1 → well-conditioned softmax inputs.

**Causal masking**: In language modeling (decoder), position i must not attend
to positions j > i (future tokens). We enforce this by setting those score
positions to −∞ before softmax, so their weight becomes 0 after softmax.

## 5. Implementation: Using ScaledDotProductAttention

In [ ]:
# Instantiate the module
attention = ScaledDotProductAttention(dropout=0.0).to(device)
print(attention)

# Forward pass (no mask)
output, weights = attention(Q, K, V, mask=None)

print(f"\nOutput shape  : {output.shape}   (batch, heads, seq_q, d_v)")
print(f"Weights shape : {weights.shape}  (batch, heads, seq_q, seq_k)")
print(f"Weights sum   : {weights[0, 0].sum(dim=-1)}  (each row should sum to 1)")

In [ ]:
# Compare SCALED vs UNSCALED attention weights
# Unscaled: compute raw scores without dividing by sqrt(d_k)

def raw_attention_weights(Q, K):
    """Return attention weights WITHOUT scaling."""
    scores = torch.matmul(Q, K.transpose(-2, -1))  # no 1/sqrt(d_k)
    return F.softmax(scores, dim=-1)

def scaled_attention_weights(Q, K, d_k):
    """Return attention weights WITH scaling."""
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    return F.softmax(scores, dim=-1)

unscaled_w = raw_attention_weights(Q, K)    # [batch, heads, seq_q, seq_k]
scaled_w   = scaled_attention_weights(Q, K, D_K)

# Measure entropy: high entropy = more spread out attention (usually better)
def entropy(w):
    """Mean entropy of attention distribution across all queries."""
    return -(w * (w + 1e-9).log()).sum(dim=-1).mean().item()

print("Effect of scaling on attention distribution:")
print(f"  Unscaled entropy : {entropy(unscaled_w):.4f}")
print(f"  Scaled   entropy : {entropy(scaled_w):.4f}")
print(f"  Unscaled max weight: {unscaled_w.max().item():.4f}")
print(f"  Scaled   max weight: {scaled_w.max().item():.4f}")
print("\n(Higher entropy = more evenly spread attention = less saturation)")

In [ ]:
# Build and apply a causal mask
# Causal mask: position i can only attend to j <= i
# True = "mask this out" (will become -inf before softmax)

def make_causal_mask(seq_len: int) -> torch.Tensor:
    """Upper-triangular boolean mask. True = future position (mask out)."""
    # Upper triangle EXCLUDING diagonal → shape [seq_len, seq_len]
    mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)
    # Expand to [1, 1, seq_len, seq_len] for broadcasting over batch and heads
    return mask.unsqueeze(0).unsqueeze(0)

causal_mask = make_causal_mask(SEQ_LEN).to(device)
print(f"Causal mask shape: {causal_mask.shape}")
print("Causal mask (True = masked = future):")
print(causal_mask[0, 0].int())

# Apply mask in forward pass
masked_output, masked_weights = attention(Q, K, V, mask=causal_mask)
print(f"\nMasked output shape  : {masked_output.shape}")
print(f"Masked weights shape : {masked_weights.shape}")
print("\nAttention weights for batch=0, head=0 (should be lower-triangular):")
print(masked_weights[0, 0].detach().cpu().numpy().round(3))

## 6. Sanity Checks

In [ ]:
# Shape checks
assert output.shape  == (BATCH_SIZE, NUM_HEADS, SEQ_LEN, D_V), f"{output.shape}"
assert weights.shape == (BATCH_SIZE, NUM_HEADS, SEQ_LEN, SEQ_LEN), f"{weights.shape}"
print("Shape checks passed.")

# Weights sum to 1 across key dimension
row_sums = weights.sum(dim=-1)
assert torch.allclose(row_sums, torch.ones_like(row_sums), atol=1e-5), \
    f"Attention weights don't sum to 1: {row_sums}"
print("Attention weights sum-to-1 check passed.")

# Causal mask check: upper triangle should be zero
upper_triangle = masked_weights[0, 0].triu(diagonal=1)
assert upper_triangle.abs().max().item() < 1e-6, \
    "Causal mask not applied correctly — future positions have non-zero weight!"
print("Causal mask check passed (no future positions attended to).")

# No NaN values
assert not torch.isnan(output).any(), "NaN in output!"
assert not torch.isnan(masked_output).any(), "NaN in masked output!"
print("No NaN values. All sanity checks passed!")

## 7. Visualization

In [ ]:
# Plot: scaled vs unscaled attention weights (batch=0, head=0)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

tok_labels = [f't{i}' for i in range(SEQ_LEN)]

def plot_attn_heatmap(ax, weights_tensor, title):
    w = weights_tensor[0, 0].detach().cpu().numpy()
    im = ax.imshow(w, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(SEQ_LEN))
    ax.set_yticks(range(SEQ_LEN))
    ax.set_xticklabels(tok_labels)
    ax.set_yticklabels(tok_labels)
    ax.set_xlabel('Key position')
    ax.set_ylabel('Query position')
    ax.set_title(title)
    for i in range(SEQ_LEN):
        for j in range(SEQ_LEN):
            ax.text(j, i, f'{w[i,j]:.2f}', ha='center', va='center',
                    color='white' if w[i,j] > 0.5 else 'black', fontsize=8)
    return im

plot_attn_heatmap(axes[0], unscaled_w,     f'Unscaled (d_k={D_K})')
plot_attn_heatmap(axes[1], scaled_w,       f'Scaled   (÷√{D_K}={math.sqrt(D_K):.2f})')
plot_attn_heatmap(axes[2], masked_weights, 'Scaled + Causal Mask')

fig.suptitle('Scaled Dot-Product Attention: Unscaled vs Scaled vs Masked', fontsize=13)
fig.tight_layout()

plot_path = RUNS_ATTENTION_DIR / 'scaled_dot_product_heatmap.png'
fig.savefig(plot_path, dpi=120)
plt.close(fig)
print(f"Heatmap saved to: {plot_path}")

In [ ]:
# Plot: causal mask heatmap separately
fig, ax = plt.subplots(figsize=(5, 4))
w_masked = masked_weights[0, 0].detach().cpu().numpy()
im = ax.imshow(w_masked, cmap='Greens', vmin=0, vmax=1)
ax.set_xticks(range(SEQ_LEN))
ax.set_yticks(range(SEQ_LEN))
ax.set_xticklabels(tok_labels)
ax.set_yticklabels(tok_labels)
ax.set_title('Masked Attention Weights\n(upper triangle = zero)')
ax.set_xlabel('Key (past) position')
ax.set_ylabel('Query position')
plt.colorbar(im, ax=ax)
fig.tight_layout()

mask_path = RUNS_ATTENTION_DIR / 'masked_attention_heatmap.png'
fig.savefig(mask_path, dpi=120)
plt.close(fig)
print(f"Masked heatmap saved to: {mask_path}")

In [ ]:
# Save shape info to CSV
shape_rows = [
    {'tensor': 'Q', 'shape': str(tuple(Q.shape)), 'description': 'Query [batch, heads, seq_q, d_k]'},
    {'tensor': 'K', 'shape': str(tuple(K.shape)), 'description': 'Key   [batch, heads, seq_k, d_k]'},
    {'tensor': 'V', 'shape': str(tuple(V.shape)), 'description': 'Value [batch, heads, seq_k, d_v]'},
    {'tensor': 'scores', 'shape': f'({BATCH_SIZE}, {NUM_HEADS}, {SEQ_LEN}, {SEQ_LEN})', 'description': 'Raw scores QK^T'},
    {'tensor': 'weights', 'shape': str(tuple(weights.shape)), 'description': 'Softmax(scores/sqrt(d_k))'},
    {'tensor': 'output', 'shape': str(tuple(output.shape)), 'description': 'Weighted sum of V'},
    {'tensor': 'causal_mask', 'shape': str(tuple(causal_mask.shape)), 'description': 'Upper-triangular bool mask'},
]
csv_path = RUNS_ATTENTION_DIR / 'attention_shapes.csv'
save_table_csv(shape_rows, csv_path)
print(f"Shape info saved to: {csv_path}")
for row in shape_rows:
    print(f"  {row['tensor']:<14} {row['shape']:<35} {row['description']}")

## 8. Failure Cases

**Softmax of −∞ row**: If ALL keys are masked for a given query (e.g., the first token in an encoder with a fully-masking padding mask), softmax produces NaN. The `ScaledDotProductAttention` implementation guards against this with `nan_to_num(..., nan=0.0)`.

**Forgetting to scale**: Try D_K=512 without scaling — notice the entropy drops close to 0, attention concentrates on one token per query. This makes training very hard.

**Wrong mask direction**: `True` means mask-out in this codebase (set to −∞). If you accidentally invert the mask, future tokens get the most attention.

## 9. Exercises

1. **Large d_k experiment**: Repeat with D_K=64. Measure the entropy of unscaled vs scaled weights. How does the gap change?
2. **Dropout effect**: Instantiate `ScaledDotProductAttention(dropout=0.5)`. Run in `.train()` mode and observe that weights no longer sum to 1 (dropout kills some weights).
3. **Padding mask**: Create a mask that masks out positions 3 and 4 (simulating padding). Verify those columns in the weight matrix are zero.
4. **Self-attention vs cross-attention**: Use Q from sequence A (length 3) and K, V from sequence B (length 7). What are the expected output and weight shapes?
5. **Manual formula**: Implement scaled dot-product attention using only `torch.matmul`, `torch.softmax`, and `torch.sqrt` — without the module. Compare output to the module.

## 10. Key Takeaways

- **Scaled dot-product attention** divides scores by √d_k to keep softmax well-conditioned.
- **Shape contract**: Q,K,V come in as `[batch, heads, seq, d]` and output is `[batch, heads, seq_q, d_v]`.
- **Attention weights** always sum to 1 per query (after softmax, ignoring dropout).
- **Causal mask** sets future positions to −∞ before softmax → zero weight → autoregressive decoding.
- This single operation is all you need — multi-head attention just applies it in parallel with different projections.